# Socioeconomic Inequality in Disaster Damage

This notebook is the complete real-data socioeconomic analysis. It does not call `socio_analysis.py`, create synthetic data, or invent missing values.

**Research question:** Is building damage associated with socioeconomic conditions after accounting for real hazard exposure and built-environment controls?

## 1. Analysis Plan

1. Validate real XBD, TIGER/Line, ACS, and hazard-exposure inputs.
2. Load post-disaster building labels and assign buildings to census units.
3. Aggregate damage and join socioeconomic variables.
4. Clean the census sentinel values, then fit a naive, a built-environment-controlled, and an
   event-fixed-effects regression.
5. Save tables, a summary, and diagnostic figures (income gradient, coefficient forest).

## 2. Real Inputs

Update the paths in the next cell. The notebook requires real XBD, TIGER/Line, and ACS files; it will stop with a clear error when one is absent.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from shapely.geometry import shape
from IPython.display import display, Image

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'data').exists():
    REPO_ROOT = Path('..').resolve()

SPLITS = REPO_ROOT / 'data' / 'splits.csv'
XBD_DIR = REPO_ROOT / 'xbd'
CENSUS_TRACTS = REPO_ROOT / 'data' / 'census_tracts.geojson'
ACS = REPO_ROOT / 'data' / 'acs.csv'
OUTPUT_DIR = REPO_ROOT / 'results' / 'socio_analysis'
EVENTS = []
MIN_BUILDINGS = 20
GEO_LEVEL = 'block_group'

print(f'Repository: {REPO_ROOT}')
print(f'Output directory: {OUTPUT_DIR}')

Repository: C:\Users\bhvns\Documents\University of Konstanz\Courses\satellite-disaster-damage-mapping
Output directory: C:\Users\bhvns\Documents\University of Konstanz\Courses\satellite-disaster-damage-mapping\results\socio_analysis


## 3. Validate Inputs

This cell checks paths, schemas, event names, and the real variables required by the analysis.

In [ ]:
required_paths = {'splits': SPLITS, 'xbd_dir': XBD_DIR, 'census_tracts': CENSUS_TRACTS, 'acs': ACS}
missing = [f'{name}: {path}' for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing real analysis inputs:\n' + '\n'.join(missing) + '\nSet the paths in the configuration cell.')

splits_df = pd.read_csv(SPLITS)
if 'event_name' not in splits_df.columns and 'disaster' in splits_df.columns:
    splits_df = splits_df.rename(columns={'disaster': 'event_name'})
acs_df = pd.read_csv(ACS)
acs_df['GEOID'] = acs_df['GEOID'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(12)

required_splits = {'scene_id', 'event_name'}
required_acs = {'GEOID', 'income_2017', 'pov_rate_2017', 'median_home_value_2017', 'renter_pct_2017', 'median_year_built_2017', 'pop_density_2017'}
if required_splits - set(splits_df.columns):
    raise ValueError(f'splits.csv missing columns: {sorted(required_splits - set(splits_df.columns))}')
if required_acs - set(acs_df.columns):
    raise ValueError(f'ACS data missing columns: {sorted(required_acs - set(acs_df.columns))}')

allowed_events = {'hurricane-harvey', 'hurricane-michael', 'hurricane-florence', 'midwest-flooding', 'socal-fire', 'woolsey-fire', 'santa-rosa-wildfire', 'joplin-tornado', 'moore-tornado', 'tuscaloosa-tornado'}
selected_events = set(EVENTS) if EVENTS else set(splits_df['event_name'].dropna()) & allowed_events
unknown_events = selected_events - allowed_events
if unknown_events:
    raise ValueError(f'Events are not in the US-event whitelist: {sorted(unknown_events)}')
splits_df = splits_df[splits_df['event_name'].isin(selected_events)].copy()
print(f'Scenes: {len(splits_df):,}')
print(f'Events: {sorted(selected_events)}')
print(f'ACS rows: {len(acs_df):,}')

Scenes: 38,080
Events: ['hurricane-florence', 'hurricane-harvey', 'hurricane-michael', 'joplin-tornado', 'midwest-flooding', 'moore-tornado', 'santa-rosa-wildfire', 'socal-fire', 'tuscaloosa-tornado', 'woolsey-fire']
ACS/exposure rows: 217,739


## 4. Load and Spatially Join Buildings

Each building is represented by the centroid of its real post-disaster label polygon and joined to a TIGER/Line census unit.

In [13]:
from shapely import wkt

tracts_gdf = gpd.read_file(CENSUS_TRACTS)
if tracts_gdf.crs is None:
    raise ValueError('Census boundaries have no CRS; provide WGS84 or a declared CRS.')
tracts_gdf = tracts_gdf.to_crs('EPSG:4326')
if 'GEOID' not in tracts_gdf.columns:
    raise ValueError('Census boundaries must contain GEOID.')
tracts_gdf['GEOID'] = tracts_gdf['GEOID'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(12)

records = []
for row in splits_df.itertuples(index=False):
    candidates = []
    if getattr(row, 'subdir', '') == 'labels' and getattr(row, 'kind', '') == 'post_disaster':
        candidates.append(REPO_ROOT / Path(str(row.path).replace('\\', '/')))
    candidates.extend([XBD_DIR / row.event_name / 'labels' / f'{row.scene_id}_post_disaster.json', XBD_DIR / 'labels' / f'{row.scene_id}_post_disaster.json', XBD_DIR / f'{row.scene_id}_post_disaster.json'])
    label_path = next((path for path in candidates if path.exists()), None)
    if label_path is None:
        continue
    with label_path.open(encoding='utf-8') as handle:
        data = json.load(handle)
    features = data.get('features', [])
    if isinstance(features, dict):
        features = features.get('lng_lat', [])
    for index, feature in enumerate(features):
        properties = feature.get('properties', {})
        if feature.get('geometry'):
            geometry = shape(feature['geometry'])
        elif feature.get('wkt'):
            geometry = wkt.loads(feature['wkt'])
        else:
            continue
        records.append({'geometry': geometry.centroid, 'uid': properties.get('uid', f'{row.scene_id}_{index}'), 'damage_class': properties.get('subtype', 'unknown'), 'scene_id': row.scene_id, 'event_name': row.event_name})
if not records:
    raise ValueError('No real buildings were loaded from the configured XBD labels.')

buildings = gpd.GeoDataFrame(records, geometry='geometry', crs='EPSG:4326')
buildings = gpd.sjoin(buildings, tracts_gdf[['GEOID', 'geometry']], how='inner', predicate='within').drop(columns=['index_right'])
if buildings.empty:
    raise ValueError('Spatial join produced no buildings inside the census boundaries.')
print(f'Buildings loaded and joined: {len(buildings):,}')

Buildings loaded and joined: 189,642


## 5. Aggregate Damage and Join Socioeconomic Data

Damage scores are: no damage = 0, minor = 1, major = 2, destroyed = 3. Census units below the configured building threshold are excluded.

In [ ]:
damage_map = {'no-damage': 0, 'minor-damage': 1, 'major-damage': 2, 'destroyed': 3}
buildings['damage_score'] = buildings['damage_class'].map(damage_map)
buildings = buildings.dropna(subset=['damage_score']).copy()

grouped = (buildings.groupby('GEOID')
           .agg(building_count=('uid', 'count'),
                damage_score_mean=('damage_score', 'mean'),
                damage_score_std=('damage_score', 'std'))
           .reset_index())

# Add the share of buildings in each damage class, one column per class.
for damage_class in damage_map:
    hit = (buildings['damage_class'] == damage_class).astype(int)
    share = (buildings.assign(hit=hit)
             .groupby('GEOID')['hit'].mean()
             .rename(f'share_{damage_class}'))
    grouped = grouped.merge(share, on='GEOID', how='left')

grouped = grouped[grouped['building_count'] >= MIN_BUILDINGS].copy()
tracts = grouped.merge(acs_df, on='GEOID', how='inner', validate='one_to_one')

if tracts.empty:
    raise ValueError('No census units survived the building threshold and ACS join.')

# The ACS extract encodes a missing value as a large negative sentinel, not as a null. Any block
# group carrying one in a variable the regression uses is dropped rather than imputed -- filling
# it with a median would leave the sentinel's scale silently dominating the standardised
# coefficients.
ACS_MISSING = -666666666
for column in ['income_2017', 'median_home_value_2017', 'median_year_built_2017', 'pop_density_2017']:
    tracts[column] = tracts[column].replace(ACS_MISSING, np.nan)

tracts['building_density'] = tracts['pop_density_2017'] / 1000
tracts['median_building_age'] = 2020 - tracts['median_year_built_2017']

n_before_sentinel = len(tracts)
required_for_regression = ['income_2017', 'median_home_value_2017', 'renter_pct_2017',
                           'building_density', 'median_building_age']
tracts = tracts.dropna(subset=required_for_regression).copy()

# Block groups are single-event by construction (an xBD scene never spans two disasters), so the
# modal event among a block group's buildings identifies it for the event fixed-effects model.
event_of_block = buildings.groupby('GEOID')['event_name'].agg(lambda s: s.value_counts().idxmax())
tracts['event_name'] = tracts['GEOID'].map(event_of_block)

print(f'Census units before sentinel cleaning: {n_before_sentinel}')
print(f'Census units analysed: {len(tracts):,} '
      f'({n_before_sentinel - len(tracts)} dropped for a census sentinel value)')
display(tracts.head())

## 6. Regression Analysis

Three nested specifications, all on the same 291 block groups (17 more were dropped above for
carrying a census sentinel value in at least one required variable). `pov_rate_2017` (empty for
every block group in this sample) and `exposure_proxy` (identically zero -- no fire-perimeter,
storm-track, or coastline layer was ever joined) are excluded from every specification: a
degenerate predictor contributes nothing and can distort the other standard errors.

1. **naive** -- socioeconomic variables only.
2. **+ built** -- adds building density and building age.
3. **+ event FE** -- adds one dummy per disaster event, so the comparison is between two block
   groups inside the *same* disaster. Event identity is the sharpest exposure control this data
   supports without an external hazard-exposure layer.

Standardised coefficients make effect sizes comparable across specifications.

In [ ]:
socio_features = ['income_2017', 'median_home_value_2017', 'renter_pct_2017']
control_features = ['building_density', 'median_building_age']


def fit_model(features, name, event_fe=False):
    # Standardised OLS, with the standard errors worked out by hand.
    columns = features + (['event_name'] if event_fe else []) + ['damage_score_mean']
    model_frame = tracts[columns].copy()
    model_frame[features] = model_frame[features].apply(pd.to_numeric, errors='coerce')
    model_frame = model_frame.dropna(subset=features + ['damage_score_mean'])

    predictors = StandardScaler().fit_transform(model_frame[features])
    feature_names = list(features)

    if event_fe:
        dummies = pd.get_dummies(model_frame['event_name'], drop_first=True)
        predictors = np.hstack([predictors, dummies.to_numpy(dtype=float)])
        feature_names += list(dummies.columns)

    X = np.hstack([np.ones((len(predictors), 1)), predictors])
    y = model_frame['damage_score_mean'].to_numpy()

    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    residual = y - X @ beta

    degrees_freedom = len(y) - X.shape[1]
    mse = np.sum(residual ** 2) / degrees_freedom
    covariance = np.linalg.pinv(X.T @ X) * mse
    standard_error = np.sqrt(np.maximum(np.diag(covariance), 0))

    t_value = np.divide(beta, standard_error,
                        out=np.full_like(beta, np.nan), where=standard_error > 0)
    p_value = 2 * (1 - stats.t.cdf(np.abs(t_value), degrees_freedom))

    ss_res = np.sum(residual ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r_squared = 1 - ss_res / ss_tot
    adj_r_squared = 1 - (1 - r_squared) * (len(y) - 1) / degrees_freedom

    result = pd.DataFrame({'model': name, 'feature': ['intercept'] + feature_names, 'coef': beta,
                           'se': standard_error, 't': t_value, 'pval': p_value})
    return result[result['feature'] != 'intercept'].reset_index(drop=True), r_squared, adj_r_squared


# Each specification adds controls to the one before it, on the identical 291-block-group sample.
naive_results, naive_r2, naive_adj_r2 = fit_model(socio_features, 'naive')
built_results, built_r2, built_adj_r2 = fit_model(control_features + socio_features, 'plus_built')
fe_results, fe_r2, fe_adj_r2 = fit_model(control_features + socio_features, 'plus_event_fe',
                                         event_fe=True)
regression_results = pd.concat([naive_results, built_results, fe_results], ignore_index=True)

print(f'n = {len(tracts):,} block groups')
print(f'naive       R2={naive_r2:.4f}  adj R2={naive_adj_r2:.4f}')
print(f'+ built     R2={built_r2:.4f}  adj R2={built_adj_r2:.4f}')
print(f'+ event FE  R2={fe_r2:.4f}  adj R2={fe_adj_r2:.4f}')
display(regression_results[regression_results['feature'].isin(socio_features + control_features)])

## 7. Results and Figures

The notebook saves the complete analysis outputs under `OUTPUT_DIR` (`results/socio_analysis/`)
and displays them here: the three-specification regression table, a coefficient-forest plot
showing that the two socioeconomic effects shrink toward zero once event fixed effects enter while
the two built-environment effects do not, and an income-quintile / within-event-income-tercile view
of the raw damage score that shows the same pattern without fitting a model at all.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
buildings.drop(columns='geometry').to_csv(OUTPUT_DIR / 'buildings.csv', index=False)
tracts.to_csv(OUTPUT_DIR / 'tracts.csv', index=False)
regression_results.to_csv(OUTPUT_DIR / 'regression_results.csv', index=False)

summary = '\n'.join(["=== Socioeconomic Damage Analysis ===",
                     f"Events: {sorted(selected_events)}",
                     f"Buildings analysed: {len(buildings)}",
                     f"Census units analysed: {len(tracts)} (of {n_before_sentinel} before sentinel cleaning)",
                     f"Minimum buildings per unit: {MIN_BUILDINGS}",
                     f"naive R2: {naive_r2:.4f}",
                     f"+ built R2: {built_r2:.4f}",
                     f"+ event FE R2: {fe_r2:.4f}"])
(OUTPUT_DIR / 'summary.txt').write_text(summary, encoding='utf-8')

# --- Figure 1: income gradient, pooled across events and within a single event -----------------
tracts['quintile'] = pd.qcut(tracts['income_2017'], 5, labels=[f'Q{i}' for i in range(1, 6)])
quintiles = tracts.groupby('quintile', observed=True)['damage_score_mean'].agg(['mean', 'sem'])

event_counts = tracts['event_name'].value_counts()
big_events = event_counts[event_counts >= 15].index
within_event = tracts[tracts['event_name'].isin(big_events)].copy()
within_event['tercile'] = within_event.groupby('event_name')['income_2017'].transform(
    lambda s: pd.qcut(s.rank(method='first'), 3, labels=['low', 'mid', 'high']))
terciles = (within_event.groupby('tercile', observed=True)['damage_score_mean']
            .agg(['mean', 'sem']).reindex(['low', 'mid', 'high']))

quintiles.to_csv(OUTPUT_DIR / 'income_quintiles.csv')
terciles.to_csv(OUTPUT_DIR / 'within_event_terciles.csv')

fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.4))
axes[0].bar(range(5), quintiles['mean'], yerr=1.96 * quintiles['sem'], capsize=3, color='#2a78d6')
axes[0].set_xticks(range(5), quintiles.index)
axes[0].set_xlabel('block-group income quintile (pooled across events)')
axes[0].set_ylabel('mean damage score')
axes[0].set_title('Pooled across events: U-shaped in income', loc='left', fontsize=10)

axes[1].bar(range(3), terciles['mean'], yerr=1.96 * terciles['sem'], capsize=3, color='#eb6834')
axes[1].set_xticks(range(3), ['low', 'mid', 'high'])
axes[1].set_xlabel('within-event income tercile')
axes[1].set_ylabel('mean damage score')
axes[1].set_title('Within each event: gradient flattens', loc='left', fontsize=10)

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'socio_income_gradient.png', dpi=150)
plt.close(fig)

# --- Figure 2: coefficient forest across all three specifications ------------------------------
spec_order = ['naive', 'plus_built', 'plus_event_fe']
spec_label = {'naive': 'naive', 'plus_built': '+ built', 'plus_event_fe': '+ event FE'}
socio_vars = ['income_2017', 'median_home_value_2017']
built_vars = ['building_density', 'median_building_age']
var_label = {'income_2017': 'Median income', 'median_home_value_2017': 'Median home value',
            'building_density': 'Building density', 'median_building_age': 'Building age'}
rows = socio_vars + built_vars
y_base = {v: i for i, v in enumerate(rows)}
offsets = {'naive': 0.22, 'plus_built': 0.0, 'plus_event_fe': -0.22}
colors = {'naive': '#c3c2b7', 'plus_built': '#5598e7', 'plus_event_fe': '#1c5cab'}

fig, ax = plt.subplots(figsize=(7.2, 4.2))
for spec in spec_order:
    spec_frame = regression_results[regression_results['model'] == spec].set_index('feature')
    for var in rows:
        if var not in spec_frame.index:
            continue
        coef, se = spec_frame.loc[var, 'coef'], spec_frame.loc[var, 'se']
        ax.errorbar(coef, y_base[var] + offsets[spec], xerr=1.96 * se, fmt='o',
                    color=colors[spec], capsize=3, ms=5)

ax.axvline(0, color='black', lw=0.8)
ax.set_yticks(range(len(rows)), [var_label[v] for v in rows])
ax.set_xlabel('standardised coefficient (95% CI)')
ax.invert_yaxis()
ax.spines[['top', 'right']].set_visible(False)
handles = [plt.Line2D([0], [0], marker='o', color=colors[s], linestyle='', label=spec_label[s])
          for s in spec_order]
ax.legend(handles=handles, loc='lower right', frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'socio_coefficients.png', dpi=150)
plt.close(fig)

print(summary)
display(pd.read_csv(OUTPUT_DIR / 'regression_results.csv'))
display(Image(filename=str(OUTPUT_DIR / 'socio_income_gradient.png')))
display(Image(filename=str(OUTPUT_DIR / 'socio_coefficients.png')))

## 8. Interpretation and Limitations

The naive and `+ built` specifications show a significant, positive association between income and
damage. Adding event fixed effects -- comparing block groups inside the *same* disaster -- drops
the income coefficient to statistical insignificance while $R^2$ more than triples: most of the
apparent income effect is between-event variation (which disaster struck which kind of place), not
a within-disaster inequality gradient. The income-quintile and within-event-tercile view of the raw
damage score shows the same pattern without fitting a model: pooled across events the relationship
is U-shaped -- a mix of low-income tornado exposure and high-income coastal / wildland-urban-interface
exposure -- and it flattens once income is ranked inside a single event. The two built-environment
effects (building density, building age) are the only ones that survive event fixed effects.

These are associations, not causal effects, and describe block groups, not individual households.
Conclusions depend on the temporal and geographic match between XBD labels, TIGER boundaries, and
ACS measures. See `notebooks/damage_cost_analysis.ipynb` for what this means once damage is priced
in dollars and checked against real recovery-capacity data (insurance settlements, FEMA grants) --
the picture there is considerably less equal than the raw damage score alone suggests.